# Niknam et al. (2019) on CRCNS mt-3

Paper: Niknam, Akbarian, Clark, Zamani, Noudoost and Nategh (2019), "Characterizing and dissociating
multiple time-varying modulatory computations influencing neuronal activity", PLoS Comput Biol,
doi:10.1371/journal.pcbi.1007275. The claims (`claims/niknam-2019.yaml`) were written from the
version of record (PMC6759185), its figure legends and the legends of its supplement.

Dataset: CRCNS mt-3 (doi:10.6080/K0FB514J), `mt-3/live/data.zip` in the public bucket
`crcnsarchive`, streamed and opened in place. CRCNS data sets are not versioned, so the version is
recorded as `unversioned` (docs/archives.md). The bucket holds one copy of the data (`live/`, no
`old/`), so there is no other version to choose; the data descriptor is version 0.6 of
2019-08-06, the only one, and describes these files. MD5 sums from the data set's `checksums.md5`:
`data.zip` a5f225a0b73d64b5e2d2f881e345752a (checked below against the bytes read),
`docs/crcns_mt-3_data_description.pdf` 0a3515c7d3a4a8aae3b0aded6daa3d7a.

Outside input (`with_labels`): `assets/data/main_lists.mat` of the paper's code,
https://github.com/nnategh/SFA-Models at commit 1438dde9dd3207348ec0df1e5c4dd2a7b014e68e, for
each neuron's RF, FF and ST probe (`probe_list`). Its other tables are shown for reference only.

| Claim | Tested here |
|---|---|
| probe-locations-link | tested |
| methods-neuron-counts | tested |
| fig1b-saccadic-suppression, fig1c-ff-remapping, fig1d-st-remapping, fig1e-ff-and-st-remapping | tested |
| fig1-remapping-longer-latency, fig1-example-neurons | tested |
| fig3, fig5a to fig5c, fig6, fig7, fig8, figs4, figs5 (25 claims) | blocked: they need the S-, F-, A- or N-model. The S-model's parameter selection as ported here, timed below, needs hundreds of hours on the runner's 4 CPUs, and the full fits, the ten S-model fits of the A-model and the simulations come on top (`too_large`); fig5a-n-model-no-perisaccadic-gain is blocked because the N-model (ref [48]) was not identified when it was tested (O8); ref [48] has since been identified, but this attempt did not implement the N-model |
| figs1-rf-centres | blocked: no grid geometry in the data set |

In [ ]:
import hashlib, io, time, resource, collections
import numpy as np
import pandas as pd
import fsspec
import matplotlib.pyplot as plt
from scipy import stats
import mt3

T0 = time.time()
zf = mt3.open_zip()
names = mt3.neuron_names(zf)
N = [mt3.load_neuron(zf, n) for n in names]
lists = mt3.load_probe_list()
probe_list = lists["probe_list"]
row_of = {(int(r[0]), int(r[1])): r for r in probe_list}
print(f"loaded {len(N)} neurons in {time.time() - T0:.1f} s, "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

What was loaded, by session, and every variable the analysis uses, against the data
descriptor (`paper/mt-3.data-description.txt`, version 0.6) and the paper's Methods.

In [ ]:
# MD5 of data.zip against the archive's checksums.md5 (the zip is read into memory, not to disk)
with fsspec.open(mt3.CHECKSUMS_URL, "r") as f:
    md5s = dict(line.split()[::-1] for line in f.read().strip().splitlines())
with fsspec.open(mt3.DATA_URL, "rb", block_size=2**20) as f:
    zip_md5 = hashlib.md5(f.read()).hexdigest()
print("data.zip md5", zip_md5, "checksums.md5:", md5s["data.zip"], "match:", zip_md5 == md5s["data.zip"])

rows = []
for d in N:
    sc, r = d["stimcode"], d["resp"]
    rows.append(dict(neuron=d["name"], session=d["session"], channel=d["channel"],
                     trials=sc.shape[0], ms_per_trial=sc.shape[1], conditions=len(np.unique(d["conds"])),
                     spikes=int(r.sum()), rate_spk_s=r.sum() / r.size * 1000,
                     tsaccade_min=d["tsaccade"].min(), tsaccade_max=d["tsaccade"].max(),
                     stim_min=int(sc.min()), stim_max=int(sc.max()), resp_values=str(np.unique(r).tolist())))
inv = pd.DataFrame(rows)
pd.set_option("display.width", 200, "display.max_columns", 30)
print(inv.to_string(index=False))
by_session = inv.groupby("session").agg(neurons=("neuron", "size"), trials=("trials", "first"))
print(by_session.to_string())
print("neurons", len(inv), "sessions", inv.session.nunique())
# descriptor: 41 neurons as 1,3,2,1,0,6,1,2,8,3,7,7 over sessions 1-12
print("neurons per session in date order:", by_session.neurons.tolist(),
      "descriptor (session 5 has 0):", [1, 3, 2, 1, 6, 1, 2, 8, 3, 7, 7])
# do neurons of a session share stimuli, conditions and saccade times?
for s, g in inv.groupby("session"):
    ds = [d for d in N if d["session"] == s]
    same = all((x["stimcode"] == ds[0]["stimcode"]).all() and (x["tsaccade"] == ds[0]["tsaccade"]).all()
               and (x["conds"] == ds[0]["conds"]).all() for x in ds)
    print(s, "stimcode, conds and tsaccade identical across its neurons:", same)

In [ ]:
# Probe timing: runs of one code (probe durations) and gaps with no probe, over all trials of one
# neuron per session; the paper says each probe lasted 7 ms with exactly one probe on screen at a time.
run_len, gap_len, first_on, last_on, cycle_ok = collections.Counter(), collections.Counter(), [], [], []
for s in inv.session.unique():
    d = next(x for x in N if x["session"] == s)
    for row in d["stimcode"]:
        ch = np.flatnonzero(np.diff(row) != 0) + 1
        starts = np.r_[0, ch]; lens = np.diff(np.r_[starts, row.size]); vals = row[starts]
        on = np.flatnonzero(vals > 0)
        run_len.update(lens[on].tolist())
        inner = np.arange(on[0], on[-1] + 1)
        gap_len.update(lens[inner][vals[inner] == 0].tolist())
        first_on.append(starts[on[0]]); last_on.append(starts[on[-1]] + lens[on[-1]])
        seq = vals[on]
        # is the probe order a repeating 81-long cycle (each code once per 81 consecutive onsets)?
        cycle_ok.append(all(len(set(seq[i:i + 81])) == min(81, len(seq[i:i + 81])) for i in range(0, len(seq) - 81, 81)))
tot = sum(run_len.values())
print("probe run lengths (ms: share of runs):", {k: round(v / tot, 4) for k, v in sorted(run_len.items())})
print("gaps between probes inside the sequence (ms: count):", dict(sorted(gap_len.items())))
print(f"first probe at {np.min(first_on)}-{np.max(first_on)} ms, last probe ends at "
      f"{np.min(last_on)}-{np.max(last_on)} ms of the 3000 ms arrays")
print("trials whose onsets run in blocks of 81 distinct codes:", np.mean(cycle_ok).round(3))

# Order of probes within a trial (bears on which pairings of O1 exist). Per session: onsets per
# trial, distinct codes per trial, presentations of each shown probe per trial, ms between repeats
# of a probe, trials where one probe falls in both the fixation (-500 to -100 ms) and the
# perisaccadic (-50 to 0 ms) window, and where the order restarts from its first probe.
seq_rows = []
for s in inv.session.unique():
    d = next(x for x in N if x["session"] == s)
    tr, t, c = mt3.all_onsets(d)
    rel = t - d["sacc"][tr]
    fixw, periw = mt3.in_window(d, tr, t, mt3.FIX), mt3.in_window(d, tr, t, mt3.PERI_LATE)
    n_on, n_codes, reps, rep_ms, both, restart = [], [], collections.Counter(), [], 0, []
    for i in range(d["stimcode"].shape[0]):
        m = tr == i
        q, ti = c[m], t[m]
        n_on.append(len(q)); n_codes.append(len(set(q)))
        reps.update(np.bincount(q)[np.unique(q)].tolist())
        for code in np.unique(q):
            rep_ms.extend(np.diff(np.sort(ti[q == code])).tolist())
        both += len(np.intersect1d(q[fixw[m]], q[periw[m]]))
        K = len(set(q))
        r = [j for j in range(1, len(q) - 3) if j % K and (q[j:j + 3] == q[:3]).all()]
        if r:
            restart.append(ti[r[0]] - d["sacc"][i])
    restart = np.array(restart)
    seq_rows.append(dict(session=s, onsets_per_trial=f"{min(n_on)}-{max(n_on)}",
                         codes_per_trial=sorted(set(n_codes)), presentations_per_probe_per_trial=dict(sorted(reps.items())),
                         ms_between_repeats_median=float(np.median(rep_ms)), ms_between_repeats_5pct=float(np.percentile(rep_ms, 5)),
                         probe_in_both_windows=both, trials_restarting=f"{len(restart)}/{len(n_on)}",
                         restart_ms_from_saccade=f"{restart.min()}-{restart.max()}" if len(restart) else "-"))
print(pd.DataFrame(seq_rows).to_string(index=False))
# Runs longer than 7 ms: in session 20150717, whose order never restarts, the onset sequence repeats
# exactly every 81 onsets, so a 14 or 21 ms run takes one place in the order (one probe held longer).
d = next(x for x in N if x["session"] == 20150717)
tr, t, c = mt3.all_onsets(d)
exact = all((c[tr == i][81:] == c[tr == i][:-81]).all() for i in range(d["stimcode"].shape[0]))
long_runs = sum(((np.diff(np.r_[0, np.flatnonzero(np.diff(row) != 0) + 1, row.size]) >= 13)
                 & (row[np.r_[0, np.flatnonzero(np.diff(row) != 0) + 1]] > 0)).sum() for row in d["stimcode"])
print(f"session 20150717: onset order repeats every 81 onsets in every trial: {exact}; runs of 13 ms or more: {long_runs}")


In [ ]:
# Recordings deposited more than once: identical or near-identical spike trains within a session.
pairs = []
for s in inv.session.unique():
    ds = [d for d in N if d["session"] == s]
    for i in range(len(ds)):
        for j in range(i + 1, len(ds)):
            a, b = ds[i]["resp"].ravel().astype(float), ds[j]["resp"].ravel().astype(float)
            pairs.append((ds[i]["name"], ds[j]["name"], bool((a == b).all()), np.corrcoef(a, b)[0, 1],
                          (a * b).sum() / min(a.sum(), b.sum())))
pairs = pd.DataFrame(pairs, columns=["a", "b", "identical", "r_1ms", "coincident_share"])
print(len(pairs), "pairs within sessions; identical:", int(pairs.identical.sum()))
print(pairs.sort_values("r_1ms", ascending=False).head(5).to_string(index=False))

In [ ]:
# The paper's code repository holds copies of the neuron files (assets/data); compare them with CRCNS.
diff = []
for d in N:
    raw = mt3.load_code_file(f"assets/data/{d['name']}.mat")
    import h5py
    with h5py.File(io.BytesIO(raw), "r") as h:
        same = all(np.array_equal(h[k][()].T.ravel() if k in ("conds", "tsaccade") else h[k][()].T,
                                  (d[k].ravel() if k in ("conds", "tsaccade") else d[k]).astype(h[k].dtype))
                   for k in ("stimcode", "resp", "conds", "tsaccade"))
    diff.append(same)
print("code-repository copies identical to the CRCNS files in all four variables:", sum(diff), "of", len(diff))

In [ ]:
# Clocks. Stimulus clock: the response to the listed RF probe against all other probes, by latency,
# with the true stimulus codes and with codes shifted by 300 ms. Saccade clock: the population rate
# around tsaccade, against tsaccade taken from another trial of the same session.
lags = np.arange(0, 151)
contrast_true, contrast_shift = [], []
for d in N:
    r = row_of[(d["session"], d["channel"])]
    rf = mt3.sub2ind(r[2], r[3])
    tr, t, c = mt3.all_onsets(d)
    keep = t + 300 + 151 < d["resp"].shape[1]
    tr, t, c = tr[keep], t[keep], c[keep]
    tr_ = d["resp"][tr[:, None], t[:, None] + lags]
    sh = d["resp"][tr[:, None], t[:, None] + 300 + lags]
    for out, x in ((contrast_true, tr_), (contrast_shift, sh)):
        out.append((x[c == rf].mean(0) - x[c != rf].mean(0)) * 1000)
ct, cs = np.array(contrast_true), np.array(contrast_shift)
pk = ct.mean(0).argmax()
print(f"RF minus other probes, mean over neurons: peak {ct.mean(0).max():.1f} spk/s at {pk} ms latency; "
      f"mean over 50-75 ms {ct[:, 50:76].mean():.1f} spk/s, positive in {(ct[:, 50:76].mean(1) > 0).sum()} of 41 neurons")
print(f"with codes shifted by 300 ms: mean over 50-75 ms {cs[:, 50:76].mean():.1f} spk/s, "
      f"positive in {(cs[:, 50:76].mean(1) > 0).sum()} of 41")

rng = np.random.default_rng(1)
win = np.arange(-300, 301)
mod_true, mod_shuf = [], []
for d in N:
    s = d["sacc"]
    sp = rng.permutation(len(s))
    for out, ss in ((mod_true, s), (mod_shuf, s[sp])):
        x = d["resp"][np.arange(len(ss))[:, None], ss[:, None] + win].mean(0) * 1000
        x = mt3.gauss_smooth(x, 13)
        out.append(x / x.mean())
mt, ms = np.array(mod_true).mean(0), np.array(mod_shuf).mean(0)
print(f"normalized population rate -300..300 ms around saccade: range {mt.min():.2f}-{mt.max():.2f} "
      f"(minimum at {win[mt.argmin()]} ms) with true times; {ms.min():.2f}-{ms.max():.2f} with shuffled times")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(lags, ct.mean(0), label="true codes"); ax[0].plot(lags, cs.mean(0), label="codes shifted 300 ms")
ax[0].set_xlabel("ms after probe onset"); ax[0].set_ylabel("RF minus others (spk/s)"); ax[0].legend()
ax[1].plot(win, mt, label="true tsaccade"); ax[1].plot(win, ms, label="shuffled tsaccade")
ax[1].set_xlabel("ms from saccade"); ax[1].set_ylabel("rate / mean"); ax[1].legend(); plt.show()

## probe-locations-link

> The RF, FF and ST probe of each neuron listed in the code repository's main_lists.mat
> correspond to the dataset's neurons, and the RF probe recomputed from the dataset by the
> Methods' definition matches the listed one.

Neurons are matched by session and channel (file name, `probe_list` columns 1 and 2). The RF is
recomputed by D1 (highest mean count 50-75 ms after onsets in -500 to -100 ms), onsets by O2 i.

In [ ]:
link = []
for d in N:
    key = (d["session"], d["channel"])
    r = row_of.get(key)
    rf, m = mt3.rf_from_data(d, "onset")
    rf2, _ = mt3.rf_from_data(d, "every")
    x, y = mt3.ind2sub(rf)
    x2, y2 = mt3.ind2sub(rf2)
    link.append(dict(neuron=d["name"], listed=r is not None, RF_listed=(int(r[2]), int(r[3])), RF_data=(x, y),
                     cheb=max(abs(x - r[2]), abs(y - r[3])), cheb_every=max(abs(x2 - r[2]), abs(y2 - r[3])),
                     FF_minus_RF=(int(r[4] - r[2]), int(r[5] - r[3])), ST=(int(r[6]), int(r[7])),
                     direction="left" if r[4] < r[2] else ("right" if r[4] > r[2] else "none")))
link = pd.DataFrame(link)
print(link.to_string(index=False))
print("files matched to a probe_list row:", link.listed.sum(), "of", len(N), "; probe_list rows:", len(probe_list),
      "; rows without a file:", len(set(row_of) - {(d['session'], d['channel']) for d in N}))
n_ok = int((link.cheb <= 1).sum())
print(f"recomputed RF equal to the listed RF: {(link.cheb == 0).sum()}; equal or a neighbour: {n_ok} of 41 "
      f"(every-ms reading: {(link.cheb_every <= 1).sum()})")
print("FF minus RF offsets (x, y):", collections.Counter(link.FF_minus_RF))
print("saccade direction from the sign of FF minus RF x:", link.direction.value_counts().to_dict())

**Result: reproduced.** All 41 files match a row of `probe_list` by session and channel, and the
RF recomputed from the data is the listed RF or one of its neighbours for all 41 neurons (see the
counts printed above). The FF minus RF offsets are horizontal only, and their signs give 27
leftward and 14 rightward neurons.

## methods-neuron-counts

> Of 49 well-isolated MT neurons, 41 responded to the probes and were analysed; they come from 11
> sessions, 27 neurons from 9 sessions with leftward saccades and 14 from 2 sessions with
> rightward saccades.

In [ ]:
sess_dir = link.assign(session=link.neuron.str[:8]).groupby("session").direction.agg(lambda s: s.unique().tolist())
print(sess_dir.to_string())
left = link.direction == "left"
print("neurons", len(link), "sessions", link.neuron.str[:8].nunique())
print("leftward: neurons", int(left.sum()), "sessions", link[left].neuron.str[:8].nunique())
print("rightward: neurons", int((~left).sum()), "sessions", link[~left].neuron.str[:8].nunique())

**Result: reproduced.** 41 neurons from 11 sessions; by the sign of the listed FF minus RF offset,
27 neurons from 9 sessions are leftward and 14 from 2 sessions rightward, and every session has a
single direction. The 49 isolated units cannot be counted: the 8 that were discarded are not in
the data set.

## Recorded perisaccadic effects (Figure 1)

Each neuron is classified by D2 under every reading of O1 (pairs: (a) trial, (b) condition,
(c) unpaired rank-sum, a variant) and O2 (presentations: (i) onset, (ii) every ms, a variant).
Two pairings outside O1, raised by the verifier, are shown as variants: a one-sided signed-rank
test over the 1 ms bins of the response window, on the mean count per bin (`timebin`) and on the
curves smoothed as for plots (`timebin_smoothed`). The Methods allow the first ("comparing the
perisaccadic and fixation spike counts (during the same stimulus-aligned window)") and exclude the
second ("all statistical analyses were performed on spike counts ... (not estimates of firing
rate)"). Only (a) and (b) with onsets decide (column `decides`). Reading (a) cannot be tested: in
no session does any trial show a probe in both the fixation and the perisaccadic window (data
check, `probe_in_both_windows`), as a probe returns only after a pass through the whole order,
562 to 604 ms (median per session). The outcomes therefore rest on (b).

The normalized responses follow D3; O4 (a) is the mean over the effect window of the normalized
response, averaged over neurons with SE across neurons, and O4 (b) the peak inside the window of
the neuron-averaged curve smoothed with a Gaussian of FWHM 13 ms.

In [ ]:
SPECS = {"supp": ("RF", mt3.PERI_RF, mt3.EARLY, "less"),
         "ff": ("FF", mt3.PERI_LATE, mt3.LATE, "greater"),
         "st": ("ST", mt3.PERI_LATE, mt3.LATE, "greater")}
PROBES = {}
for d in N:
    r = row_of[(d["session"], d["channel"])]
    PROBES[d["name"]] = {"RF": mt3.sub2ind(r[2], r[3]), "FF": mt3.sub2ind(r[4], r[5]), "ST": mt3.sub2ind(r[6], r[7])}
# O1 a (trial) and b (condition) decide, with O2 i (onset); the rest are variants: O1 c (unpaired),
# and two pairings outside O1, over the 1 ms bins of the response window, of mean counts (timebin)
# and of the smoothed curves (timebin_smoothed), which the Methods exclude ("not estimates of firing rate").
PAIRINGS = ("trial", "condition", "unpaired", "timebin", "timebin_smoothed")
READINGS = [(p, m) for m in ("onset", "every") for p in PAIRINGS]
DECIDES = {("trial", "onset"), ("condition", "onset")}


def run_test(d, eff, p, m):
    loc, peri, win, di = SPECS[eff]
    if p.startswith("timebin"):
        return mt3.timebin_test(d, PROBES[d["name"]][loc], peri, win, di, m, smooth=p == "timebin_smoothed")
    return mt3.effect_test(d, PROBES[d["name"]][loc], peri, win, di, p, m)


TESTS = {(d["name"], eff, p, m): run_test(d, eff, p, m) for d in N for eff in SPECS for p, m in READINGS}
sig, TESTABLE = {}, {}
rows = []
for p, m in READINGS:
    TESTABLE[(p, m)] = any(not np.isnan(TESTS[(d["name"], "ff", p, m)]["p"]) for d in N)
    for eff in SPECS:
        ps = np.array([TESTS[(d["name"], eff, p, m)]["p"] for d in N])
        sig[(eff, p, m)] = np.nan_to_num(ps, nan=1.0) < 0.05
    row = dict(pairing=p, presentations=m, decides=(p, m) in DECIDES,
               pairs_median=np.median([TESTS[(d["name"], "ff", p, m)]["n"] if p != "unpaired" else 0 for d in N]),
               neurons_testable=sum(not np.isnan(TESTS[(d["name"], "ff", p, m)]["p"]) for d in N))
    for eff, lab in (("supp", "suppression"), ("ff", "ff"), ("st", "st")):
        row[lab] = int(sig[(eff, p, m)].sum()) if TESTABLE[(p, m)] else "not testable"
    row["ff_and_st"] = int((sig[("ff", p, m)] & sig[("st", p, m)]).sum()) if TESTABLE[(p, m)] else "not testable"
    rows.append(row)
counts_tab = pd.DataFrame(rows)
print("Neurons significant by D2 under each reading (paper: 8, 23, 37, 22)")
print(counts_tab.to_string(index=False))
# number of presentations per neuron in each window, onset reading
npres = pd.DataFrame([{**{"neuron": d["name"]}, **{f"{e}_{k}": TESTS[(d["name"], e, "condition", "onset")][k]
                       for e in SPECS for k in ("n_fix", "n_peri")}} for d in N])
print(npres.describe().loc[["min", "50%", "max"]].round(1).to_string())


In [ ]:
def population(names_sel, loc, peri, win, mode="onset"):
    """D3 normalized responses of the selected neurons; O4 a and b."""
    fs, ps = [], []
    for d in N:
        if d["name"] not in names_sel:
            continue
        F, P = mt3.mean_trains(d, PROBES[d["name"]][loc], peri, mode)
        f, p = mt3.normalized(F, P)
        fs.append(f); ps.append(p)
    fs, ps = np.array(fs), np.array(ps)
    if len(fs) == 0:
        return dict(n=0)
    wf = fs[:, win[0]:win[1] + 1].mean(1); wp = ps[:, win[0]:win[1] + 1].mean(1)
    sf, sp = mt3.gauss_smooth(fs.mean(0)), mt3.gauss_smooth(ps.mean(0))
    return dict(n=len(fs), fix_a=wf.mean(), fix_a_se=wf.std(ddof=1) / np.sqrt(len(wf)) if len(wf) > 1 else np.nan,
                peri_a=wp.mean(), peri_a_se=wp.std(ddof=1) / np.sqrt(len(wp)) if len(wp) > 1 else np.nan,
                fix_b=sf[win[0]:win[1] + 1].max(), peri_b=sp[win[0]:win[1] + 1].max(), curves=(fs, ps))


def within(v, e, tol):
    return abs(v - e) <= tol + 1e-9


def fig1_table(eff, loc, peri, win, expected, selector=None):
    rows = []
    for p, m in READINGS:
        row = dict(pairing=p, presentations=m, decides=(p, m) in DECIDES)
        if not TESTABLE[(p, m)]:
            row["count"] = "not testable"
            rows.append(row)
            continue
        sel = sig[(eff, p, m)] if selector is None else selector(p, m)
        names_sel = {d["name"] for d, s in zip(N, sel) if s}
        pop = population(names_sel, loc, peri, win, m)
        row["count"] = int(sel.sum())
        if pop["n"]:
            row.update(fix_a=pop["fix_a"], fix_a_se=pop["fix_a_se"], peri_a=pop["peri_a"], peri_a_se=pop["peri_a_se"],
                       fix_b=pop["fix_b"], peri_b=pop["peri_b"])
            row["in_tol_a"] = [within(row["count"], expected[0][0], expected[0][1]), within(row["fix_a"], *expected[1]),
                               within(row["peri_a"], *expected[2])]
        rows.append(row)
    return pd.DataFrame(rows)


### fig1b-saccadic-suppression

> Eight neurons show significant saccadic suppression; for them the average normalized response
> to an RF probe drops from 2.53 to 1.51 when the probe appears just before the saccade.

In [ ]:
t1b = fig1_table("supp", "RF", mt3.PERI_RF, mt3.EARLY, [(8, 0.8), (2.53, 0.37), (1.51, 0.29)])
print("expected: count 8 (tol 0.8), fixation 2.53 (0.37), perisaccadic 1.51 (0.29)")
print(t1b.round(3).to_string(index=False))
sel = sig[("supp", "condition", "onset")]
print("suppressed neurons under (b, i):", [d["name"] for d, s in zip(N, sel) if s])
print("by session:", collections.Counter(d["name"][:8] for d, s in zip(N, sel) if s))
for d, s in zip(N, sel):
    if s:
        t = TESTS[(d["name"], "supp", "condition", "onset")]
        print(f"  {d['name']}: conditions {t['n']}, mean count fixation {t['mean_fix']:.3f}, perisaccadic {t['mean_peri']:.3f}, p {t['p']:.4f}")
pop = population({d["name"] for d, s in zip(N, sel) if s}, "RF", mt3.PERI_RF, mt3.EARLY)
fs, ps = pop["curves"]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(mt3.gauss_smooth(fs.mean(0)), label="fixation"); ax.plot(mt3.gauss_smooth(ps.mean(0)), label="perisaccadic")
ax.axvspan(50, 75, alpha=.1); ax.set_xlabel("ms after RF probe onset"); ax.set_ylabel("normalized response"); ax.legend(); plt.show()
print(f"plotted range: fixation {mt3.gauss_smooth(fs.mean(0)).min():.2f}-{mt3.gauss_smooth(fs.mean(0)).max():.2f}, "
      f"perisaccadic {mt3.gauss_smooth(ps.mean(0)).min():.2f}-{mt3.gauss_smooth(ps.mean(0)).max():.2f}")

**Result: reproduced under O1 (b), pairing by condition, with onsets (O2 i).** Eight neurons are significant (paper 8), and their normalized RF response falls from 2.235 to 1.239 by O4 (a) (paper 2.53 ± 0.37 to 1.51 ± 0.29; both within tolerance, near its lower edge); by O4 (b) the peaks are 2.541 and 1.530. Reading (a), pairing by trial, cannot be tested, because no trial shows the same probe in both windows. The 8 neurons are only partly the paper's: the classification agrees with the code's `nuron_list` on 35 of 41 neurons (reference section), so 5 of the 8 are in that list and 3 are not. The variants give 4 to 16 neurons; the unpaired test with onsets gives 9 with both values within tolerance, the time-bin tests 13 and 15.

### fig1c-ff-remapping

> Twenty-three neurons show significant FF-remapping; for them the average normalized response
> to an FF probe rises from 0.90 during fixation to 1.30 just before the saccade.

In [ ]:
t1c = fig1_table("ff", "FF", mt3.PERI_LATE, mt3.LATE, [(23, 2.3), (0.90, 0.03), (1.30, 0.08)])
print("expected: count 23 (tol 2.3), fixation 0.90 (0.03), perisaccadic 1.30 (0.08)")
print(t1c.round(3).to_string(index=False))
sel = sig[("ff", "condition", "onset")]
print("FF-remapping neurons under (b, i):", [d["name"] for d, s in zip(N, sel) if s])
print("by direction:", collections.Counter(link.direction[sel]))

**Result: not reproduced.** Under O1 (b) with onsets only 8 neurons show FF-remapping (paper 23, tolerance 2.3), and their normalized FF response rises from 0.792 to 1.609 (paper 0.90 to 1.30); the rise is present but none of the three numbers is within tolerance. Reading (a) cannot be tested. No variant reaches the count: they give 9 to 19 neurons. The closest, the time-bin test on smoothed curves with onsets (19 neurons, 0.885 to 1.411) and the unpaired, time-bin and smoothed time-bin tests with every-ms presentations (17 neurons, 0.888 or 0.889 to 1.427 or 1.430), have only the fixation value within tolerance. The time-bin test on mean counts with onsets gives 10 neurons. 7 of the 8 neurons under (b) are from leftward sessions.

### fig1d-st-remapping

> Thirty-seven neurons show significant ST-remapping; for them the average normalized late
> response to a probe near the saccade target rises from 0.83 during fixation to 1.35 just before
> the saccade.

In [ ]:
t1d = fig1_table("st", "ST", mt3.PERI_LATE, mt3.LATE, [(37, 3.7), (0.83, 0.03), (1.35, 0.06)])
print("expected: count 37 (tol 3.7), fixation 0.83 (0.03), perisaccadic 1.35 (0.06)")
print(t1d.round(3).to_string(index=False))
sel = sig[("st", "condition", "onset")]
print("ST-remapping neurons under (b, i):", [d["name"] for d, s in zip(N, sel) if s])
print("by direction:", collections.Counter(link.direction[sel]))

**Result: not reproduced.** Under O1 (b) with onsets 11 neurons show ST-remapping (paper 37, tolerance 3.7), and their normalized ST response rises from 0.749 to 1.629 (paper 0.83 to 1.35); none of the three numbers is within tolerance. Reading (a) cannot be tested. The outcome depends on the pairing. The time-bin test on smoothed curves with onsets gives 34 neurons, 0.815 to 1.365, all three numbers within tolerance, and agrees with the code's `nuron_list` on ST for 36 of 41 neurons, against 15 for (b) (reference section). The Methods exclude that test, as it compares firing-rate estimates, and the claims file does not list it, so it decides nothing here; whether O1 should hold a pairing over time bins is asked of a reader (`questions/niknam-2019.yaml`). On mean counts per bin, the time-bin test gives 16 neurons with onsets (no number within tolerance) and 29 with every-ms presentations (0.819 to 1.383, both values within tolerance); the unpaired test with every-ms presentations gives 31 (0.823 to 1.369).

### fig1e-ff-and-st-remapping

> Twenty-two neurons show both FF- and ST-remapping; for them the normalized FF response rises
> from 0.90 to 1.32 and the normalized ST response from 0.80 to 1.46.

In [ ]:
rows = []
for p, m in READINGS:
    row = dict(pairing=p, presentations=m, decides=(p, m) in DECIDES)
    if not TESTABLE[(p, m)]:
        row["count"] = "not testable"
        rows.append(row)
        continue
    sel = sig[("ff", p, m)] & sig[("st", p, m)]
    names_sel = {d["name"] for d, s in zip(N, sel) if s}
    row["count"] = int(sel.sum())
    if names_sel:
        a = population(names_sel, "FF", mt3.PERI_LATE, mt3.LATE, m)
        b = population(names_sel, "ST", mt3.PERI_LATE, mt3.LATE, m)
        row.update(ff_fix=a["fix_a"], ff_peri=a["peri_a"], st_fix=b["fix_a"], st_peri=b["peri_a"],
                   ff_fix_b=a["fix_b"], ff_peri_b=a["peri_b"], st_fix_b=b["fix_b"], st_peri_b=b["peri_b"])
        exp = [(22, 2.2), (0.90, 0.03), (1.32, 0.08), (0.80, 0.03), (1.46, 0.08)]
        vals = [row["count"], row["ff_fix"], row["ff_peri"], row["st_fix"], row["st_peri"]]
        row["n_in_tol"] = sum(within(v, *e) for v, e in zip(vals, exp))
        row["both_increase"] = bool(row["ff_peri"] > row["ff_fix"] and row["st_peri"] > row["st_fix"])
    rows.append(row)
t1e = pd.DataFrame(rows)
print("expected: count 22 (2.2); FF 0.90 (0.03) to 1.32 (0.08); ST 0.80 (0.03) to 1.46 (0.08)")
print(t1e.round(3).to_string(index=False))


**Result: not reproduced.** Under O1 (b) with onsets 6 neurons show both effects (paper 22), and none of the five numbers is within tolerance (FF 0.777 to 1.680, ST 0.750 to 1.789), though both responses rise. Reading (a) cannot be tested. The variants give 6 to 19 neurons, none within tolerance of 22. The time-bin test on smoothed curves with onsets (19 neurons) and the unpaired, time-bin and smoothed time-bin tests with every-ms presentations (17 neurons) each have 3 of 5 numbers within tolerance, which would meet the partial rule; the others have none.

### fig1-remapping-longer-latency

> Perisaccadic FF and ST responses come at a longer latency (80-150 ms after the probe) than RF
> responses (50-75 ms).

Peak latency of the neuron-averaged normalized response 0-150 ms after onset, smoothed with a
Gaussian of FWHM 13 ms: fixation RF response over all 41 neurons; perisaccadic FF response of the
FF-remapping neurons and perisaccadic ST response of the ST-remapping neurons, under each reading.

In [ ]:
allnames = {d["name"] for d in N}
rf_pop = population(allnames, "RF", mt3.PERI_RF, mt3.EARLY, "onset")
rf_curve = mt3.gauss_smooth(rf_pop["curves"][0].mean(0))
print(f"RF fixation peak (41 neurons, onset reading): {rf_curve.argmax()} ms")
rf_every = int(mt3.gauss_smooth(population(allnames, "RF", mt3.PERI_RF, mt3.EARLY, "every")["curves"][0].mean(0)).argmax())
rows = []
for p, m in READINGS:
    row = dict(pairing=p, presentations=m, decides=(p, m) in DECIDES)
    row["rf_peak"] = int(rf_curve.argmax()) if m == "onset" else rf_every
    if not TESTABLE[(p, m)]:
        row["n_ff"] = row["n_st"] = "not testable"
        rows.append(row)
        continue
    for eff, loc in (("ff", "FF"), ("st", "ST")):
        names_sel = {d["name"] for d, s in zip(N, sig[(eff, p, m)]) if s}
        row[f"n_{eff}"] = len(names_sel)
        if names_sel:
            c = mt3.gauss_smooth(population(names_sel, loc, mt3.PERI_LATE, mt3.LATE, m)["curves"][1].mean(0))
            row[f"{eff}_peri_peak"] = int(c.argmax())
    rows.append(row)
tlat = pd.DataFrame(rows)
print(tlat.to_string(index=False))
# per-neuron variant: peak of each remapping neuron's smoothed perisaccadic response (onset, condition)
for eff, loc in (("ff", "FF"), ("st", "ST")):
    pk = []
    for d, s in zip(N, sig[(eff, "condition", "onset")]):
        if s:
            F, P = mt3.mean_trains(d, PROBES[d["name"]][loc], mt3.PERI_LATE)
            pk.append(int(mt3.gauss_smooth(P.mean(0)).argmax()))
    print(f"{eff}: per-neuron perisaccadic peaks {sorted(pk)}")


**Result: reproduced.** The neuron-averaged fixation RF response of the 41 neurons peaks at 62 ms after onset, inside 50-75 ms, and the perisaccadic FF and ST responses of the remapping neurons peak at 108 and 99 ms under O1 (b) with onsets, inside 80-150 ms. Every testable reading gives the same order (RF 59-62 ms, FF 103-108 ms, ST 96-128 ms). The remapped peaks are not an independent test: the neurons are selected for an increase 80-150 ms after onset, which pulls the averaged perisaccadic peak into that window, and the peaks of single neurons are spread (FF 12 to 150 ms, ST 0 to 150 ms). The RF peak, over all 41 neurons with no selection, is the part of the result that does not depend on the selection.

### fig1-example-neurons

> In example neurons the RF response falls from 60.89 to 40.49 spk/s before the saccade; the late
> FF response rises from 7.90 to 18.42 spk/s; the late ST response rises from 23.80 to 64.64 spk/s;
> and in a neuron with both effects FF rises from 15.83 to 39.10 and ST from 16.15 to 36.17 spk/s,
> each with p < 0.001.

Rates are mean rates over the effect window after presentations in the D2 windows, SE across
presentations. Every neuron is searched for each example.

In [ ]:
EX = {"RF example": [("supp", "RF", 60.89, 2.82, 40.49, 12.64)],
      "FF example": [("ff", "FF", 7.90, 1.02, 18.42, 3.56)],
      "ST example": [("st", "ST", 23.80, 1.77, 64.64, 7.91)],
      "FF and ST example": [("ff", "FF", 15.83, 0.91, 39.10, 5.14), ("st", "ST", 16.15, 0.94, 36.17, 4.49)]}
ex_rows = []
for mode in ("onset", "every"):
    for d in N:
        for ex, parts in EX.items():
            ok_all, desc = True, []
            for eff, loc, ef, tf, ep, tp in parts:
                _, peri, win, di = SPECS[eff]
                F, P = mt3.mean_trains(d, PROBES[d["name"]][loc], peri, mode)
                rf_, rp_ = mt3.window_rate(F, win), mt3.window_rate(P, win)
                pa = TESTS[(d["name"], eff, "condition", mode)]["p"]
                pt = TESTS[(d["name"], eff, "trial", mode)]["p"]
                okv = within(rf_.mean(), ef, tf) and within(rp_.mean(), ep, tp)
                oks = (pa < 0.05) or (pt < 0.05)
                ok_all &= okv and oks
                desc.append(f"{loc} {rf_.mean():.2f}±{rf_.std(ddof=1)/np.sqrt(len(rf_)):.2f} -> {rp_.mean():.2f}±"
                            f"{rp_.std(ddof=1)/np.sqrt(max(len(rp_),2)):.2f} (p_cond {pa:.4f}, values {'in' if okv else 'out'})")
            ex_rows.append(dict(mode=mode, example=ex, neuron=d["name"], match=ok_all, detail="; ".join(desc)))
ex_tab = pd.DataFrame(ex_rows)
for (mode, ex), g in ex_tab.groupby(["mode", "example"], sort=False):
    print(f"{mode} / {ex}: matching neurons {g[g.match].neuron.tolist()}")
# closest neurons by values only (onset reading), for each example
closest = {}
for ex, parts in EX.items():
    dist = []
    for d in N:
        tot = 0
        for eff, loc, ef, tf, ep, tp in parts:
            _, peri, win, di = SPECS[eff]
            F, P = mt3.mean_trains(d, PROBES[d["name"]][loc], peri)
            tot += abs(mt3.window_rate(F, win).mean() - ef) / tf + abs(mt3.window_rate(P, win).mean() - ep) / tp
        dist.append((tot, d["name"]))
    best = sorted(dist)[:3]
    closest[ex] = best[0][1]
    print(f"{ex}: closest (sum of |difference| / tolerance) {[(n, round(t, 2)) for t, n in best]}")
    for t, n in best[:1]:
        print("   ", ex_tab[(ex_tab["mode"] == "onset") & (ex_tab.example == ex) & (ex_tab.neuron == n)].detail.iloc[0])
# p of the D2 test under every pairing (onset presentations) for the matched neurons and the
# closest neuron of each example; the paper gives p < 0.001 for each example.
byname = {d["name"]: d for d in N}
shown = sorted(set(ex_tab[ex_tab.match].neuron) | set(closest.values()))
prow = []
for n in shown:
    for eff in SPECS:
        prow.append(dict(neuron=n, effect=eff, **{p: TESTS[(n, eff, p, "onset")]["p"] for p in PAIRINGS}))
with pd.option_context("display.float_format", "{:.2g}".format):
    print("closest neuron of each example:", closest)
    print(pd.DataFrame(prow).to_string(index=False))

# Fixation set: the fixation rate of the closest neuron of each example as the fixation window moves,
# start -560 to -440 ms and end -160 to -40 ms (1 ms steps), with the SE across presentations.
scan = []
for ex, parts in EX.items():
    n = closest[ex]; d = byname[n]
    for eff, loc, ef, tf, ep, tp in parts:
        _, peri, win, _ = SPECS[eff]
        tr, t = mt3.presentations(d, PROBES[n][loc])
        cr = mt3.counts(d, tr, t, win) * 1000.0 / (win[1] - win[0] + 1)
        rel = t - d["sacc"][tr]
        vals = {(a, b): cr[(rel >= a) & (rel <= b)].mean() for a in range(-560, -439) for b in range(-160, -39)}
        fx = (rel >= -500) & (rel <= -100)
        scan.append(dict(example=ex, neuron=n, probe=loc, paper=ef, paper_se=tf, here=cr[fx].mean(),
                         here_se=cr[fx].std(ddof=1) / np.sqrt(fx.sum()), n_fix=int(fx.sum()),
                         scan_min=min(vals.values()), scan_max=max(vals.values()), vals=vals))
scan = pd.DataFrame(scan)
print(scan.drop(columns="vals").round(2).to_string(index=False))
# one window for the three matched examples (four comparisons) at once
keys = list(scan.vals.iloc[0])
matched = scan[scan.example != "RF example"]
err = {k: max(abs(v[k] - e) for v, e in zip(matched.vals, matched.paper)) for k in keys}
best_win = sorted(err, key=err.get)[:3]
print("windows that bring the three matched examples closest to the paper (max |difference| spk/s):",
      [(k, round(err[k], 3)) for k in best_win], "; at -500 to -100:", round(err[(-500, -100)], 3))
# Saccade index one ms either way (tsaccade - 2, - 1 as used, and tsaccade): fixation and
# perisaccadic rates of the closest neurons.
off_rows = []
for ex, parts in EX.items():
    n = closest[ex]; d = byname[n]
    for eff, loc, ef, tf, ep, tp in parts:
        _, peri, win, _ = SPECS[eff]
        tr, t = mt3.presentations(d, PROBES[n][loc])
        cr = mt3.counts(d, tr, t, win) * 1000.0 / (win[1] - win[0] + 1)
        for off in (-1, 0, 1):
            rel = t - (d["sacc"][tr] + off)
            off_rows.append(dict(example=ex, probe=loc, shift_ms=off, fix=cr[(rel >= -500) & (rel <= -100)].mean(),
                                 peri=cr[(rel >= peri[0]) & (rel <= peri[1])].mean(), paper_fix=ef, paper_peri=ep))
print(pd.DataFrame(off_rows).round(2).to_string(index=False))

**Result: partially reproduced.** With onsets (O2 i), neurons match three of the four examples on both values and on a significant D2 test under O1 (b): 2015071702 for the FF example (7.83 to 18.96 spk/s, p 0.017), 2015092707 for the ST example (23.61 to 64.64, p 0.0059) and 2015051115 for the example with both effects (FF 15.76 to 39.10, p 0.00063; ST 16.05 to 36.17, p 0.00037). Their perisaccadic means equal the paper's to the second decimal in three of the five comparisons. No neuron matches the RF example. The closest, 2015092709, has the paper's perisaccadic value (40.49 ± 12.64) but a fixation rate of 56.94 ± 2.71 against 60.89 ± 2.82, and p 0.49 under (b) (0.02 unpaired, 0.015 on time bins); moving the fixation window (start -560 to -440 ms, end -160 to -40 ms) gives it 53.87 to 59.68, never 60.89. With every-ms presentations (a variant) 2015083106 matches the RF example.

The paper's p < 0.001 holds under (b) only for the example with both effects. Under the unpaired test the three matched examples give p 0.00011 or less, and under the time-bin test on mean counts 1.1e-09 or less except the FF example (0.023). The fixation values of the three matched examples differ from the paper's by 0.07 to 0.19 spk/s while their SEs agree within 0.01 (for example 1.01 against 1.02), so the paper's fixation sets are about the same size as ours. No window in the scan matches all three at once: the closest, -495 to -100 ms, is within 0.054 spk/s, against 0.194 at -500 to -100 ms. Shifting the saccade index by 1 ms either way moves the perisaccadic rates away from the paper's (40.49 becomes 44.87 or 33.94) and the fixation rates by 0.11 spk/s at most. The paper's fixation set is not identified.

## Model claims: the cost of the S-model

Every claim from Figure 3 on needs the S-model fitted per neuron (D4), and most need the F-model,
the A-model (ten S- and F-model fits per neuron) and 1000 simulated stimulus sequences per model.
The S-model's parameter selection (Methods, S-model framework; `select_bases.m`) fits, for every
probe (81), time basis and delay basis, a single-parameter model on 100 random 65% subsets and on
100 subsets with shuffled responses. Below, that fit, ported from `select_bases.m`, is timed on one
neuron and one probe for a few time bases, and the time for the whole selection is estimated.

In [ ]:
rng_bench = np.random.default_rng(7)
d = N[0]
Bd = mt3.bspline_bases(150)
Bt = mt3.bspline_bases(1081)
print("delay bases", Bd.shape[0], "time bases", Bt.shape[0], "-> parameters per probe", Bd.shape[0] * Bt.shape[0],
      "and per neuron", 81 * Bd.shape[0] * Bt.shape[0])
times = np.arange(-540, 541)
probe = PROBES[d["name"]]["RF"]
stim = (d["stimcode"] == probe).astype(float)
nt = stim.shape[0]
BS = np.zeros((Bd.shape[0], nt, len(times)))
for j in range(Bd.shape[0]):
    for i in range(nt):
        cv = np.convolve(Bd[j], stim[i])
        BS[j, i] = cv[d["sacc"][i] + times]
rsp = np.array([d["resp"][i, d["sacc"][i] + times] for i in range(nt)], float)
b0 = -np.log(1000.0 / (d["resp"][d["stimcode"] > 0].mean() * 1000) - 1)
conds = d["conds"]; uc = np.unique(conds)
n_fit, t_fit = 0, 0.0
for tb in (40, 78, 120):
    idx = Bt[tb] > 0
    for j in range(Bd.shape[0]):
        for shuffle in (False, True):
            tr_c = rng_bench.permutation(uc)
            ntr = int(len(uc) * 0.35); nval = int((len(uc) - ntr) * 0.30 / 0.65)
            trn = np.flatnonzero(np.isin(conds, tr_c[:ntr])); val = np.flatnonzero(np.isin(conds, tr_c[ntr:ntr + nval]))
            t0 = time.time()
            mt3.select_fit(BS[j][:, idx], rsp[:, idx], b0, trn, val, rng_bench, shuffle)
            t_fit += time.time() - t0; n_fit += 1
per_fit = t_fit / n_fit
fits_neuron = 81 * Bd.shape[0] * Bt.shape[0] * 200
trials_all = sum(x["resp"].shape[0] for x in N)
est_h = per_fit * fits_neuron / 3600 * trials_all / nt
print(f"{n_fit} fits in {t_fit:.1f} s ({per_fit * 1000:.1f} ms per fit, {nt} trials)")
# Lower bound for a fully batched port: only samples where the regressor is nonzero change with the
# parameter, and each gradient step of select_bases.m evaluates the likelihood at least 4 times
# (train and validation, before and after), at least 2 steps per fit. Per-element cost measured on
# one large vectorized evaluation.
nz = []
for tb in range(0, Bt.shape[0], 5):
    idx = Bt[tb] > 0
    nz.extend((BS[j][:, idx] != 0).sum() * 0.65 for j in range(Bd.shape[0]))
nz_mean = float(np.mean(nz))
xx = rng_bench.random(10_000_000); rr = (rng_bench.random(10_000_000) < 0.01).astype(float)
t0 = time.time()
lam = 1000.0 / (1.0 + np.exp(-(0.3 * xx - 4.0))); _ = np.sum(rr * np.log(lam * 1e-3) - lam * 1e-3)
per_elem = (time.time() - t0) / 1e7
lb_h = fits_neuron * nz_mean * 8 * per_elem / 3600 * trials_all / nt
print(f"nonzero samples per fit (65% of trials): mean {nz_mean:.0f}; per-element likelihood cost {per_elem * 1e9:.1f} ns")
print(f"lower bound for the selection of all 41 neurons, fully batched: {lb_h:.1f} CPU hours, {lb_h / 4:.1f} hours on 4 CPUs")
# The A-model fits the S-model ten times on 65% subsets (Methods, A-model), selection included.
print(f"with the ten S-model fits of the A-model, fully batched: {10 * lb_h:.0f} CPU hours, {10 * lb_h / 4:.0f} hours on 4 CPUs")
print(f"selection alone: {fits_neuron:,} fits per neuron; all 41 neurons ({trials_all} trials), scaled by trials: "
      f"{est_h:,.0f} CPU hours, {est_h / 4:,.0f} hours on 4 CPUs")

**Result: not tested; the model claims are blocked as too large.** The S-model has 290,628 stimulus-basis parameters per neuron, and its selection step runs 58,125,600 single-parameter fits per neuron. This port of `select_bases.m` took 2.2 to 6.7 ms per fit in the runs of this attempt (the value printed above is this run's), that is 345 to 1,040 hours on 4 CPUs for the selection step of the 41 neurons. That port is what this attempt has, and it is far over the notebook's 60 minutes (CI fails a notebook over sixty) and the five hours of an author run. A fully batched port, not written here, would evaluate only the nonzero samples; its lower bound printed above was 11.5 to 27.5 CPU hours (2.9 to 6.9 hours on 4 CPUs) across runs, for one selection per neuron. It is a lower bound for the selection step only. The full S-model fit, the F-model fits and the 1000 simulated sequences come on top, and the A-model fits the S-model ten times per neuron (Methods, A-model: "fitting the S-model ten times as explained earlier"), which multiplies the selection bound by ten, 29 to 69 hours on 4 CPUs. Five claims also need the N-model. The paper defines it only through ref [48], which this revision identified as Akbarian et al. (2018), IEEE Trans Biomed Eng 65(2):241-253, doi:10.1109/TBME.2017.2762687 (open as PMC5796416). No code for it is published, and this attempt did not implement it.

## figs1-rf-centres

The RF centres in dva need each session's grid position and probe spacing. The files hold probe
codes on the 9 x 9 grid only (data check above); the descriptor gives no geometry, and the paper
gives only ranges (Methods, paragraph 1). The claim is blocked (`metadata_insufficient`).

## Reference: the paper's own classification lists

`main_lists.mat` in the code repository holds, besides `probe_list`, five undocumented 41 x 6
tables (session, channel, four 0/1 columns): `nuron_list`, `nsglm_list`, `twobm_list`,
`gauss_list` and `aggat_list`. No code reads them. Their counts are compared with the paper and
with the classification of this notebook; they decide no outcome.

In [ ]:
for k in ("nuron_list", "nsglm_list", "twobm_list", "gauss_list", "aggat_list"):
    v = lists[k]
    assert all((int(a), int(b)) in row_of for a, b in v[:, :2])
    print(f"{k}: column sums {v[:, 2:].sum(0).tolist()}, columns 4 and 6 both {(v[:, 3] & v[:, 5]).sum()}, "
          f"columns 5 and 6 both {(v[:, 4] & v[:, 5]).sum()}")
nl = lists["nuron_list"]
key = {(d["session"], d["channel"]): i for i, d in enumerate(N)}
order = [key[(int(a), int(b))] for a, b in nl[:, :2]]
for p, m in READINGS:
    if not TESTABLE[(p, m)]:
        print(f"agreement with nuron_list ({p}, {m}): not testable")
        continue
    s_ = [sig[(e, p, m)][order] for e in SPECS]
    print(f"agreement with nuron_list ({p}, {m}): suppression {(s_[0] == nl[:, 2]).sum()}/41, "
          f"FF (column 4) {(s_[1] == nl[:, 3]).sum()}/41, ST (column 6) {(s_[2] == nl[:, 5]).sum()}/41; "
          f"listed ST neurons found {(s_[2] & (nl[:, 5] == 1)).sum()}/{nl[:, 5].sum()}")
for k in ("aggat_list", "gauss_list", "twobm_list", "nsglm_list"):
    v = lists[k]
    print(f"{k}: neurons with the effect in nuron_list and in the list, columns 3-6: "
          f"{[(int((nl[:, j] & v[:, j]).sum())) for j in range(2, 6)]}")
# Per neuron, by name: nuron_list columns 3 (suppression), 4 (FF) and 6 (ST), and the one-sided p of
# D2 under each pairing with onset presentations (cond = O1 b, unp = O1 c, tb = time bins of mean
# counts, tbs = time bins of smoothed curves).
nl_row = {(int(r[0]), int(r[1])): r for r in nl}
pn = []
for d in N:
    r = nl_row[(d["session"], d["channel"])]
    row = dict(neuron=d["name"], list_supp=int(r[2]), list_ff=int(r[3]), list_st=int(r[5]))
    for e in SPECS:
        for p, lab in (("condition", "cond"), ("unpaired", "unp"), ("timebin", "tb"), ("timebin_smoothed", "tbs")):
            row[f"{e}_{lab}"] = TESTS[(d["name"], e, p, "onset")]["p"]
    pn.append(row)
pn = pd.DataFrame(pn)
with pd.option_context("display.float_format", "{:.2g}".format, "display.width", 250):
    print(pn.to_string(index=False))


## Summary

| Claim | Outcome |
|---|---|
| probe-locations-link | reproduced |
| methods-neuron-counts | reproduced |
| fig1b-saccadic-suppression | reproduced |
| fig1c-ff-remapping | not reproduced |
| fig1d-st-remapping | not reproduced |
| fig1e-ff-and-st-remapping | not reproduced |
| fig1-remapping-longer-latency | reproduced |
| fig1-example-neurons | partially reproduced |
| fig5a-n-model-no-perisaccadic-gain | blocked (methods underspecified: the N-model) |
| 24 other claims on the S-, F- and A-models (Figures 3, 5 to 8, S4, S5) | blocked (too large) |
| figs1-rf-centres | blocked (metadata insufficient) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The Methods name a one-sided signed-rank test without
saying what is paired. Pairing by trial (O1 a) is impossible in these data, because no trial shows
the same probe in both windows, so the outcomes rest on pairing by condition (O1 b). Under that
reading saccadic suppression matches the paper (8 neurons, 5 of them in the code's `nuron_list`)
but FF- and ST-remapping are found in far fewer neurons (8 and 11 against 23 and 37). Variants the
claims file does not let decide come closer: the unpaired test on every millisecond a probe is on
gives 12, 17 and 31, and a signed-rank test over the 1 ms bins of the smoothed response curves,
which the Methods exclude, gives 15, 19 and 34, agreeing with `nuron_list` on ST for 36 of 41
neurons. Whether the claims file should include a pairing over time bins is asked of a reader. The
example neurons' fixation rates differ slightly from the paper's while their perisaccadic rates
match, so the paper's fixation set also differs from -500 to -100 ms in a way not identified here.
No model was fitted. The N-model's reference [48] is Akbarian et al. (2018), IEEE Trans Biomed
Eng 65(2):241-253, identified after the N-model claim was verified as blocked; a later attempt can
implement it from that paper.

What was found about the dataset. The 41 files match the paper's 41 neurons and 11 sessions and
the descriptor's per-session counts, and they are identical to the copies in the paper's code
repository. Each recording appears once. The stimulus and saccade clocks hold: the RF response
peaks at 58 ms after probe onset and disappears with codes shifted by 300 ms. The files do not
give the saccade direction, the RF, FF or ST probes, the grid's position and spacing, or which
monkey each session comes from; the probes are only in the code repository. Probes do not tile
time as the paper says: about 91.5% of probe runs last 7 ms and 5.6% last 6 ms; most others last
14 or 21 ms, and a few 8 to 28 ms. Between probes there are gaps of 7 to 28 ms with no probe on
screen, nearly all of 10, 11, 20 or 21 ms. A 14 or 21 ms run takes one place in the probe order. The
order does not hold all 81 probes in three sessions (75 in 20150511, 80 in 20150927 and 20150929),
and in every session but 20150717 it restarts from its first probe 35 to 296 ms after saccade
onset in most trials, so each probe is shown 2 to 4 times per trial. The arrays are 3000 ms long,
while the paper gives trials of 2100 to 2300 ms.

In [ ]:
print(f"total run time {time.time() - T0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")